# NB03 - Manipulation et Transformations des données avec Databricks SQL

## Mise en place

Exécuter la cellules suivantes pour s'assurer créer / rafraichir les différents éléments du **Catalogue**

In [0]:
%run "./Resources/NB00/Setup"

Vérifier que l'on utilise le bon **Catalogue** et **Schéma** pour éviter de devoir renseigner le chemin complet dans les requêtes.

`SELECT current_catalog(), current_database();`

et pour changer les catalogues et schémas par défaut utiliser la requête : 

```sql
USE CATALOG labuser_<username>;
USE SCHEMA sql_analytics;
```

In [0]:
%sql
SELECT current_catalog(), current_database();

In [0]:
%sql
USE CATALOG labuser_<username>;
USE SCHEMA sql_analytics;

## Exploration

Pour explorer on peut soit utiliser le catalogue ou le faire par SQL comme décris dans les notebooks précédents.
Avec `SHOW TABLES` on peut récupérer la listes des tables présentes dans le schéma séléctionné et si on veut plus d'informations sur une table en particulier on peut exécuter la requête suivante : 

`DESCRIBE EXTENDED TABLE ca_orders;`

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
DESCRIBE EXTENDED ca_orders;

## Manipulation et transformations des données

1. Renommage de colonnes
2. Standardisation d'une colonne
3. Décoder une date en plusieurs éléments
4. Création de variable catégorique a partir des valeurs d'une colonne
5. JOIN
6. Création d'une nouvelle table **Silver**

In [0]:
%sql
SELECT * FROM ca_orders LIMIT 10;

### 1. Rennomage d'une colonne

En faisant le `SELECT` précédent on voit une colonne **salesrep_id** et on va renommer cette colonne en **sales_rep_id** :

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  sales_channel,
  order_status
FROM ca_orders
LIMIT 10;

### 2. Standardisation d'une colonne

Mettre les valeurs des colonnes **sales_channel** et **order_status** en majuscule :

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,   --renommage de la colonne
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  upper(sales_channel) as sales_channel,
  upper(order_status) as order_status
FROM ca_orders
LIMIT 10;

### 3. Décodage d'une date

Dans la colonne **order_date** on a une date au format *yyyy-MM-dd* et on va à partir de cette colonne généré 3 nouvelles colonnes : 
- order_year
- order_month
- order_day
- order_quarter

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  upper(sales_channel) as sales_channel,
  upper(order_status) as order_status,
  year(order_date) as order_year,
  month(order_date) as order_month,
  day(order_date) as order_day,
  quarter(order_date) as order_quarter
FROM ca_orders
LIMIT 10;

### 4. Création d'une colonne catégorique

On va créer une colonne **order_amount_size** qui va représenter la taille d'une commande.
On va dans un premier temps récupérer les terciles liés à la colonne **amount** avec : 

```sql
SELECT
    percentile_cont(0.3333) WITHIN GROUP (ORDER BY amount) AS seuil_petite_moyenne,
    percentile_cont(0.6667) WITHIN GROUP (ORDER BY amount) AS seuil_moyenne_grande
FROM ca_orders;
```

Pour définir les conditions suivantes : 
- Si < seuil_petite_moyenne = PETITE
- Si > seuil_petite_moyenne et < seuil_moyenne_grande = MOYENNE
- Sinon GRANDE

In [0]:
SELECT
    percentile_cont(0.3333) WITHIN GROUP (ORDER BY amount) AS seuil_petite_moyenne,
    percentile_cont(0.6667) WITHIN GROUP (ORDER BY amount) AS seuil_moyenne_grande
FROM ca_orders;

In [0]:
SELECT
  order_line_id,
  order_id,
  order_date,
  customer_id,
  product_id,
  salesrep_id as sales_rep_id,
  quantity,
  unit_price,
  discount_rate,
  amount,
  currency,
  upper(sales_channel) as sales_channel,
  upper(order_status) as order_status,
  year(order_date) as order_year,
  month(order_date) as order_month,
  day(order_date) as order_day,
  quarter(order_date) as order_quarter,
  CASE
    WHEN amount < percentile_cont(0.3333) WITHIN GROUP (ORDER BY amount) OVER() THEN 'SMALL'
    WHEN amount < percentile_cont(0.6666) WITHIN GROUP (ORDER BY amount) OVER() THEN 'MEDIUM'
    ELSE 'LARGE'
  END as order_amount_size
FROM ca_orders
LIMIT 10;

### 5. JOIN

On va remplacer les **sales_rep_id** par le nom du représentant en faisant un **join** sur la table `ca_sales_reps`
et le **product_id** par **product_name** en faisant un **join** sur la table `ca_products`

In [0]:
select * from ca_sales_reps limit 10;

In [0]:
select * from ca_products limit 10;

In [0]:
SELECT
  o.order_line_id,
  o.order_id,
  o.order_date,
  o.customer_id,
  o.product_id,
  p.product_name as product_name,
  o.salesrep_id as sales_rep_id,
  r.salesrep_name as sales_rep_name,
  o.quantity,
  o.unit_price,
  o.discount_rate,
  o.amount,
  o.currency,
  upper(o.sales_channel) as sales_channel,
  upper(o.order_status) as order_status,
  year(o.order_date) as order_year,
  month(o.order_date) as order_month,
  day(o.order_date) as order_day,
  quarter(o.order_date) as order_quarter,
  CASE
    WHEN o.amount < percentile_cont(0.3333) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'SMALL'
    WHEN o.amount < percentile_cont(0.6666) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'MEDIUM'
    ELSE 'LARGE'
  END as order_amount_size
FROM ca_orders as o
INNER JOIN ca_sales_reps as r
ON o.salesrep_id = r.salesrep_id
INNER JOIN ca_products as p
ON o.product_id = p.product_id
LIMIT 10;

### 6. Création d'une table **Silver**

Maintenant que nous disposons d'une table contenant les formats, les colonnes et les jointures souhaités, nous allons créer la table **Silver** `ca_orders_silver`, qui contiendra les données nettoyées et enrichies.

In [0]:
create or replace table ca_orders_silver as
SELECT
  o.order_line_id,
  o.order_id,
  o.order_date,
  o.customer_id,
  o.product_id,
  p.product_name as product_name,
  o.salesrep_id as sales_rep_id,
  r.salesrep_name as sales_rep_name,
  o.quantity,
  o.unit_price,
  o.discount_rate,
  o.amount,
  o.currency,
  upper(o.sales_channel) as sales_channel,
  upper(o.order_status) as order_status,
  year(o.order_date) as order_year,
  month(o.order_date) as order_month,
  day(o.order_date) as order_day,
  quarter(o.order_date) as order_quarter,
  concat(order_year, 'Q', order_quarter) as order_year_quarter,
  CASE
    WHEN o.amount < percentile_cont(0.3333) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'SMALL'
    WHEN o.amount < percentile_cont(0.6666) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'MEDIUM'
    ELSE 'LARGE'
  END as order_amount_size
FROM ca_orders as o
INNER JOIN ca_sales_reps as r
ON o.salesrep_id = r.salesrep_id
INNER JOIN ca_products as p
ON o.product_id = p.product_id;

In [0]:
select * from ca_orders_silver;

In [0]:
DESCRIBE EXTENDED ca_orders_silver;

Si on va dans le catalogue, peut aller voir les détails de la nouvelle table et comment elle est créée / alimentée avec son **lineage** :

![image_1790869482980.png](./Resources/image_1790869482980.png "image_1790869482980.png")

## Analyse

Maintenant qu'on a une table enrichie avec toutes les données souhaitées on peut faire de l'analyse dessus.

### 1. Filtres

On va récupérer que les commandes de *Chloe Durand* triées par **amount**

In [0]:
select 
    sales_rep_name,
    order_id,
    customer_id,
    product_name,
    amount,
    order_amount_size
from ca_orders_silver
WHERE sales_rep_name = 'Chloe Durand'
order by amount desc;

### 2. Statistiques

In [0]:
select
    count(*) as total_orders,
    sum(amount) as total_sales,
    avg(amount) as avg_order_value,
    min(amount) as min_order_value,
    max(amount) as max_order_value
from ca_orders_silver;

### 3. Aggrégations et visualisation

#### 1. Les produits les plus vendus

In [0]:
select
    product_name,
    round(sum(amount), 2) as total_order_amount
from ca_orders_silver
group by product_name
order by total_order_amount desc;

Databricks visualization. Run in Databricks to view.

On a donc le resultat avec le montant total par produit et on va transformer ce résultat (tableau) en graphique.
1. Cliquer sur le flèche à côté de **Table** dans le résultat de la requête et *Visualization* : 

![image_1790930326222.png](./Resources/image_1790930326222.png "image_1790930326222.png")

2. Configurer l'onglet *General* de la manière suivante :

![image_1790930656854.png](./Resources/image_1790930656854.png "image_1790930656854.png")

3. (Optionnel) Aller dans les onglets **X Axis** et **Y Axis** pour renommer les éléments du graphique, trier les valeurs, ...

#### 2. Les produits les plus vendus par trimestre

1. Exécuter la requête
2. Configurer le graphique de la manière suivante : 

![image_1790931360234.png](./Resources/image_1790931360234.png "image_1790931360234.png")

In [0]:
select
    order_year_quarter as `Trimestre`,
    product_name as `Nom du Produit`,
    round(sum(amount), 2) as total
from ca_orders_silver
group by 
    order_year_quarter,
    product_name
order by order_year_quarter, product_name;

Databricks visualization. Run in Databricks to view.

### 4. CTEs (Common Table Expressions)

Une **CTE** (Common Table Expression) est une requête nommée et temporaire, définie avec le mot-clé `WITH`, que l'on peut réutiliser comme une table dans le `SELECT` qui suit.

**Syntaxe :**

```sql
WITH nom_cte AS (
    SELECT ... FROM ...
)
SELECT * FROM nom_cte;
```

**Avantages :**
- **Lisibilité** : décompose une requête complexe en étapes claires et nommées.
- **Réutilisabilité** : une CTE peut être référencée plusieurs fois dans la même requête.
- **Maintenabilité** : plus facile à modifier qu'une sous-requête imbriquée.

**Exemple :** calculer le chiffre d'affaires total par représentant, puis ne garder que ceux au-dessus d'un seuil.

```sql
WITH ventes_par_rep AS (
    SELECT sales_rep_name, SUM(amount) AS total_ventes
    FROM ca_orders_silver
    GROUP BY sales_rep_name
)
SELECT sales_rep_name, total_ventes
FROM ventes_par_rep
WHERE total_ventes > 100000
ORDER BY total_ventes DESC;
```



In [0]:
with totalSalesQry as (
    select round(sum(amount), 2) as total
    from ca_orders_silver
)


select
    sales_rep_name,
    round(sum(amount), 2) as sales_rep_total_sales,
    round(sales_rep_total_sales / (select total from totalSalesQry), 2) as percentage_of_sales
from ca_orders_silver
group by sales_rep_name
order by sales_rep_total_sales desc;

### 5. PIVOT

Le **PIVOT** permet de transformer des lignes en colonnes : il regroupe les valeurs d'une colonne et crée une colonne distincte pour chaque valeur, ce qui facilite la lecture croisée des données.

**Syntaxe :**

```sql
SELECT *
FROM (
    -- requête source
)
PIVOT (
    fonction_agrégation(colonne_valeur)
    FOR colonne_à_pivoter IN (valeur1, valeur2, ...)
);
```

**Avantages :**
- **Lisibilité** : présente les données sous forme de tableau croisé, plus intuitif.
- **Analyse** : permet de comparer facilement des valeurs entre plusieurs catégories.

**Exemple :** le chiffre d'affaires par représentant et par trimestre.

```sql
SELECT *
FROM (
    SELECT sales_rep_name, order_year_quarter, amount
    FROM ca_orders_silver
)
PIVOT (
    sum(amount)
    FOR order_year_quarter IN ('2023Q1', '2023Q2', '2023Q3', '2023Q4')
)
ORDER BY sales_rep_name;
```



In [0]:
SELECT *
FROM (
    SELECT sales_rep_name, order_year_quarter, amount
    FROM ca_orders_silver
)
PIVOT (
    sum(amount)
    FOR order_year_quarter IN ('2023Q1', '2023Q2', '2023Q3', '2023Q4')
)
ORDER BY sales_rep_name;

### 6. Fonction de fenêtrage

Une **fonction de fenêtrage** (window function) effectue un calcul sur un ensemble de lignes liées à la ligne courante, sans regrouper (collapse) les résultats comme le ferait un `GROUP BY`. Chaque ligne conserve donc son identité tout en bénéficiant d'une valeur agrégée ou classée.

**Syntaxe :**

```sql
fonction_agrégation(colonne)
  OVER (
    [PARTITION BY colonne]   -- sous-groupes
    [ORDER BY colonne]        -- ordre dans chaque groupe
    [frame]                   -- fenêtre coulissante (ROWS / RANGE)
  )
```

**Composantes clés :**

| Élément | Rôle |
|---|---|
| `PARTITION BY` | Divise les données en partitions indépendantes (ex: par représentant). |
| `ORDER BY` | Définit l'ordre des lignes à l'intérieur de chaque partition. |
| `Frame` (`ROWS`/`RANGE`) | Spécifie quelles lignes inclure autour de la ligne courante (ex: `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING`). |

**Fonctions courantes :**

- **Agrégations** : `SUM()`, `AVG()`, `COUNT()` … → total/moyenne sur la fenêtre.
- **Classement** : `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()` → numérotation des lignes.
- **Décalage** : `LAG()`, `LEAD()` → valeur de la ligne précédente / suivante.
- **Percentiles** : `PERCENTILE_CONT()` → valeur à un percentile donné.

**Exemple déjà utilisé dans ce notebook :**

```sql
-- Taille de commande (SMALL/MEDIUM/LARGE) basée sur les percentiles globaux
CASE
  WHEN o.amount < PERCENTILE_CONT(0.3333) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'SMALL'
  WHEN o.amount < PERCENTILE_CONT(0.6666) WITHIN GROUP (ORDER BY o.amount) OVER() THEN 'MEDIUM'
  ELSE 'LARGE'
END AS order_amount_size
```

Ici `OVER()` (sans `PARTITION BY` ni `ORDER BY`) calcule le percentile sur **toutes** les lignes de la table.

**Autre exemple — total cumulé par représentant :**

```sql
SELECT
    sales_rep_name,
    order_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY sales_rep_name
        ORDER BY order_date
    ) AS cumulative_sales
FROM ca_orders_silver
ORDER BY sales_rep_name, order_date;
```

**Différence avec `GROUP BY` :**
- `GROUP BY` réduit N lignes en 1 ligne par groupe.
- `OVER()` garde toutes les lignes et ajoute une colonne calculée.



In [0]:
select
    sales_rep_name,
    amount,
    rank() over(partition by sales_rep_name order by amount desc) as rank
from ca_orders_silver;